# Conversion of CFDBench Cylinder/prop dataset to PDEBench INS

Summary:
1) Let channel 3 be geo mask
2) Remove simulations with <20 timesteps
3) For those >20 timesteps, split into segments of 20 and concat accordingly

# Dataset Checks

In [3]:
# Cylinder/prop

import h5py
import numpy as np
import os

cfd_tube_prop = '/Volumes/T7/CFDBench/cylinder/bc'

for case in sorted(os.listdir(cfd_tube_prop)):
    case_dir = os.path.join(cfd_tube_prop, case)

    if os.path.isdir(case_dir):
        u = np.load(os.path.join(case_dir, 'u.npy'), mmap_mode='r')
        v = np.load(os.path.join(case_dir, 'v.npy'), mmap_mode='r')

        print(f"{case}: u{u.shape}, v{v.shape}")

case0000: u(620, 64, 64), v(620, 64, 64)
case0001: u(1000, 64, 64), v(1000, 64, 64)
case0002: u(1000, 64, 64), v(1000, 64, 64)
case0003: u(1000, 64, 64), v(1000, 64, 64)
case0004: u(1000, 64, 64), v(1000, 64, 64)
case0005: u(1000, 64, 64), v(1000, 64, 64)
case0006: u(1000, 64, 64), v(1000, 64, 64)
case0007: u(1000, 64, 64), v(1000, 64, 64)
case0008: u(1000, 64, 64), v(1000, 64, 64)
case0009: u(1000, 64, 64), v(1000, 64, 64)
case0010: u(1000, 64, 64), v(1000, 64, 64)
case0011: u(1000, 64, 64), v(1000, 64, 64)
case0012: u(1000, 64, 64), v(1000, 64, 64)
case0013: u(1000, 64, 64), v(1000, 64, 64)
case0014: u(1000, 64, 64), v(1000, 64, 64)
case0015: u(1000, 64, 64), v(1000, 64, 64)
case0016: u(1000, 64, 64), v(1000, 64, 64)
case0017: u(1000, 64, 64), v(1000, 64, 64)
case0018: u(1000, 64, 64), v(1000, 64, 64)
case0019: u(1000, 64, 64), v(1000, 64, 64)
case0020: u(1000, 64, 64), v(1000, 64, 64)
case0021: u(1000, 64, 64), v(1000, 64, 64)
case0022: u(1000, 64, 64), v(1000, 64, 64)
case0023: u(1

# Padding of Dataset
Based on cylinder.py:
```python
def load_case_data(case_dir: Path) -> Tuple[np.ndarray, Dict[str, float]]:
    """
    Load from the file that I have preprocessed, and pad the boundary conditions,
    turn into a numpy array of features.

    The shape of both u and v is (time steps, height, width)
    """
    case_params = load_json(case_dir / "case.json")
    # print(case_params)

    u_file = case_dir / "u.npy"
    v_file = case_dir / "v.npy"
    u = np.load(u_file)
    v = np.load(v_file)
    # Shape of u and v: (time steps, height, width)

    # Mask
    mask = np.ones_like(u)
    x_min = case_params["x_min"]
    x_max = case_params["x_max"]
    y_min = case_params["y_min"]
    y_max = case_params["y_max"]
    radius = case_params["radius"]
    case_params["center_x"] = -x_min
    case_params["center_y"] = -y_min
    for key in ["x_min", "x_max", "y_min", "y_max"]:
        del case_params[key]

    # Set the circle to 0
    height = y_max - y_min
    width = x_max - x_min
    case_params["height"] = height
    case_params["width"] = width

    dx = width / u.shape[2]
    dy = height / u.shape[1]
    for i in range(u.shape[1]):
        for j in range(u.shape[2]):
            x = x_min + j * dx
            y = y_min + i * dy
            if (x - 0.5) ** 2 + (y - 0.5) ** 2 <= radius**2:
                mask[:, i, j] = 0

    # Pad the left side
    u = np.pad(
        u,
        ((0, 0), (0, 0), (1, 0)),
        mode="constant",
        constant_values=case_params["vel_in"],
    )
    v = np.pad(v, ((0, 0), (0, 0), (1, 0)), mode="constant", constant_values=0)
    mask = np.pad(mask, ((0, 0), (0, 0), (1, 0)), mode="constant", constant_values=0)
    # # Pad the top and bottom
    u = np.pad(u, ((0, 0), (1, 1), (0, 0)), mode="constant", constant_values=0)
    v = np.pad(v, ((0, 0), (1, 1), (0, 0)), mode="constant", constant_values=0)
    mask = np.pad(mask, ((0, 0), (1, 1), (0, 0)), mode="constant", constant_values=0)

    # mask = 1 - mask
    features = np.stack([u, v, mask], axis=1)  # (T, 3, h, w)
    return features, case_params
```

In [3]:
# Padding
import numpy as np
import json
from pathlib import Path
from tqdm import tqdm
import shutil

def load_json(path):
    with open(path, 'r', encoding='utf8') as f:
        return json.load(f)

input_path = Path('/Volumes/T7/CFDBench/cylinder/prop')
output_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cylinder_prop/cylinder_prop_padded')

for case_dir in tqdm(sorted(input_path.iterdir())):

    u_file = case_dir / "u.npy"
    v_file = case_dir / "v.npy"
    u = np.load(u_file)
    v = np.load(v_file)

    case_params = load_json(case_dir / "case.json")

    mask = np.ones_like(u)

    x_min = case_params["x_min"]
    x_max = case_params["x_max"]
    y_min = case_params["y_min"]
    y_max = case_params["y_max"]
    radius = case_params["radius"]
    case_params["center_x"] = -x_min
    case_params["center_y"] = -y_min
    for key in ["x_min", "x_max", "y_min", "y_max"]:
        del case_params[key]

    height = y_max - y_min
    width = x_max - x_min
    case_params["height"] = height
    case_params["width"] = width

    dx = width / u.shape[2]
    dy = height / u.shape[1]

    for i in range(u.shape[1]):
        for j in range(u.shape[2]):
            x = x_min + j * dx
            y = y_min + i * dy
            if (x - 0.5) ** 2 + (y - 0.5) ** 2 <= radius**2:
                mask[:, i, j] = 0

    # Pad left
    u = np.pad(u, ((0, 0), (0, 0), (1, 0)), mode="constant", constant_values=case_params["vel_in"],)
    v = np.pad(v, ((0, 0), (0, 0), (1, 0)), mode="constant", constant_values=0)
    mask = np.pad(mask, ((0, 0), (0, 0), (1, 0)), mode="constant", constant_values=0)
    # # Pad the top and bottom
    u = np.pad(u, ((0, 0), (1, 1), (0, 0)), mode="constant", constant_values=0)
    v = np.pad(v, ((0, 0), (1, 1), (0, 0)), mode="constant", constant_values=0)
    mask = np.pad(mask, ((0, 0), (1, 1), (0, 0)), mode="constant", constant_values=0)
    features = np.stack([u, v, mask], axis=1)  # (T, 3, h, w)

    case_out_dir = output_path / case_dir.name
    case_out_dir.mkdir(exist_ok=True)

    np.save(case_out_dir / "features.npy", features)
    with open(case_out_dir / "case.json", "w", encoding="utf8") as f:
        json.dump(case_params, f, indent=4)

print('Done')


100%|██████████| 116/116 [00:26<00:00,  4.36it/s]

Done


In [ ]:
# Sanity Check

In [5]:
# Padded check

import h5py
import numpy as np
import os

cfd_tube_prop = '/Volumes/T7/CFDBench/Processed_experiment/cylinder_prop/cylinder_prop_padded'

for case in sorted(os.listdir(cfd_tube_prop)):
    case_dir = os.path.join(cfd_tube_prop, case)

    if os.path.isdir(case_dir):
        features = np.load(os.path.join(case_dir, 'features.npy'), mmap_mode='r')

        print(f"{case}: features{features.shape}")

case0050: features(1000, 3, 66, 65)
case0051: features(1000, 3, 66, 65)
case0052: features(1000, 3, 66, 65)
case0053: features(1000, 3, 66, 65)
case0054: features(1000, 3, 66, 65)
case0055: features(1000, 3, 66, 65)
case0056: features(1000, 3, 66, 65)
case0057: features(1000, 3, 66, 65)
case0058: features(1000, 3, 66, 65)
case0059: features(1000, 3, 66, 65)
case0060: features(1000, 3, 66, 65)
case0061: features(1000, 3, 66, 65)
case0062: features(1000, 3, 66, 65)
case0063: features(1000, 3, 66, 65)
case0064: features(1000, 3, 66, 65)
case0065: features(1000, 3, 66, 65)
case0066: features(1000, 3, 66, 65)
case0067: features(1000, 3, 66, 65)
case0068: features(1000, 3, 66, 65)
case0069: features(1000, 3, 66, 65)
case0070: features(1000, 3, 66, 65)
case0071: features(1000, 3, 66, 65)
case0072: features(1000, 3, 66, 65)
case0073: features(1000, 3, 66, 65)
case0074: features(1000, 3, 66, 65)
case0075: features(1000, 3, 66, 65)
case0076: features(1000, 3, 66, 65)
case0077: features(1000, 3, 

# Convergence check
Based on cylinder.py/CylinderFlowAutoDataset (Lines 292-308):
```python
for i in range(num_steps):
    inp = torch.tensor(inputs[i], dtype=torch.float32)  # (2, h, w)
    out = torch.tensor(outputs[i], dtype=torch.float32)

    # Check for convergence
    inp_magn = torch.sqrt(inp[0] ** 2 + inp[1] ** 2)
    out_magn = torch.sqrt(out[0] ** 2 + out[1] ** 2)
    diff = torch.abs(inp_magn - out_magn).mean()
    # print(f"Mean difference: {diff}")
    if diff < self.stable_state_diff:
        print(f"Converged at {i} out of {num_steps}, {this_case_params}")
        break
    assert not torch.isnan(inp).any()
    assert not torch.isnan(out).any()
    all_inputs.append(inp)
    all_labels.append(out)
    all_case_ids.append(case_id)
```

In [7]:
# Convergence
import numpy as np
import shutil
from pathlib import Path
from tqdm import tqdm

input_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cylinder_prop/cylinder_prop_padded')
output_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cylinder_prop/cylinder_prop_convergence')

for case_dir in tqdm(sorted(input_path.iterdir())):

    if not case_dir.is_dir():
        continue

    features_file = case_dir / "features.npy"

    features = np.load(features_file)
    num_timesteps = features.shape[0]

    converged_idx = num_timesteps - 1

    for i in range(converged_idx):
        u_in, v_in = features[i, 0, :, :], features[i, 1, :, :]
        u_out, v_out = features[i+1, 0, :, :], features[i+1, 1, :, :]

        mag_in = np.sqrt(u_in**2 + v_in**2)
        mag_out = np.sqrt(u_out**2 + v_out**2)

        diff = np.abs(mag_in - mag_out).mean()

        if diff < 1e-3:
            converged_idx = i
            break

    case_out_dir = output_path / case_dir.name
    case_out_dir.mkdir(exist_ok=True)

    truncated_features = features[:converged_idx + 1]
    np.save(case_out_dir / "features.npy", truncated_features.astype('float32'))
    shutil.copy(case_dir / "case.json", case_out_dir / "case.json")

print('Done')

100%|██████████| 116/116 [00:14<00:00,  8.09it/s]

Done


# Sanity Check

In [8]:
# Convergence check

import h5py
import numpy as np
import os

cfd_tube_prop = '/Volumes/T7/CFDBench/Processed_experiment/cylinder_prop/cylinder_prop_convergence'

for case in sorted(os.listdir(cfd_tube_prop)):
    case_dir = os.path.join(cfd_tube_prop, case)

    if os.path.isdir(case_dir):
        features = np.load(os.path.join(case_dir, 'features.npy'), mmap_mode='r')

        print(f"{case}: features{features.shape}")

case0050: features(101, 3, 66, 65)
case0051: features(136, 3, 66, 65)
case0052: features(155, 3, 66, 65)
case0053: features(167, 3, 66, 65)
case0054: features(176, 3, 66, 65)
case0055: features(182, 3, 66, 65)
case0056: features(186, 3, 66, 65)
case0057: features(187, 3, 66, 65)
case0058: features(186, 3, 66, 65)
case0059: features(184, 3, 66, 65)
case0060: features(181, 3, 66, 65)
case0061: features(177, 3, 66, 65)
case0062: features(173, 3, 66, 65)
case0063: features(169, 3, 66, 65)
case0064: features(165, 3, 66, 65)
case0065: features(160, 3, 66, 65)
case0066: features(156, 3, 66, 65)
case0067: features(153, 3, 66, 65)
case0068: features(150, 3, 66, 65)
case0069: features(146, 3, 66, 65)
case0070: features(120, 3, 66, 65)
case0071: features(101, 3, 66, 65)
case0072: features(83, 3, 66, 65)
case0073: features(50, 3, 66, 65)
case0074: features(66, 3, 66, 65)
case0075: features(79, 3, 66, 65)
case0076: features(91, 3, 66, 65)
case0077: features(101, 3, 66, 65)
case0078: features(110, 3

# Segmentation
Splits each truncated sequence after convergence step above into fixed length of 20 timesteps, discarding the remainder.

Based on convert_cfdbench.py:

```python
def split_trajectory(data_list, time_step, grid_size=64):
    traj_split = []
    for i, x in enumerate(data_list):
        T = x.shape[0]
        # num_segments = int(np.ceil(T / time_step))
        num_segments = int(np.floor(T / time_step))
        if num_segments < 1:
            continue

        padded_length = num_segments * time_step
        padded_array = x[:padded_length]
        # padded_array = np.zeros((padded_length, *x.shape[1:]))

        # Copy the original data into the padded array
        # padded_array[:T, ...] = x

        # # If needed, pad the last segment with the last frame of the original array
        # if T % time_step != 0:
        #     last_frame = x[-1, ...]
        #     padded_array[T:, ...] = last_frame

        # Reshape the array into segments
        padded_array = F.interpolate(
            torch.from_numpy(padded_array).float(), size=(grid_size, grid_size), mode="bilinear", align_corners=True
        ).numpy()
        padded_array = padded_array.reshape((num_segments, time_step, *padded_array.shape[1:]))

        traj_split.append(padded_array)

    traj_split = np.concatenate(traj_split, axis=0)
    return traj_split
```

In [9]:
# Segmentation
import numpy as np
import shutil
from pathlib import Path
from tqdm import tqdm

input_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cylinder_prop/cylinder_prop_convergence')
output_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cylinder_prop/cylinder_prop_segmented')

t_len = 20

for case_dir in tqdm(sorted(input_path.iterdir())):

    if not case_dir.is_dir():
        continue

    features_file = case_dir / "features.npy"
    json_file = case_dir / "case.json"

    features = np.load(features_file)
    num_timesteps = features.shape[0]

    num_segments = num_timesteps // t_len

    if num_segments == 0:
        continue

    for i in range(num_segments):
        start = i * t_len
        end = start + t_len

        segment = features[start:end]

        segment_name = f"{case_dir.name}_seg{i}"
        segment_out_dir = output_path / segment_name
        segment_out_dir.mkdir(exist_ok=True)

        np.save(segment_out_dir / "features.npy", segment.astype('float32'))
        shutil.copy(json_file, segment_out_dir / "case.json")

print('Done')

100%|██████████| 116/116 [00:02<00:00, 55.96it/s]

Done


# Sanity Check

In [10]:
# Segmentation check

import h5py
import numpy as np
import os

cfd_tube_prop = '/Volumes/T7/CFDBench/Processed_experiment/cylinder_prop/cylinder_prop_segmented'

for case in sorted(os.listdir(cfd_tube_prop)):
    case_dir = os.path.join(cfd_tube_prop, case)

    if os.path.isdir(case_dir):
        features = np.load(os.path.join(case_dir, 'features.npy'), mmap_mode='r')

        print(f"{case}: features{features.shape}")

case0050_seg0: features(20, 3, 66, 65)
case0050_seg1: features(20, 3, 66, 65)
case0050_seg2: features(20, 3, 66, 65)
case0050_seg3: features(20, 3, 66, 65)
case0050_seg4: features(20, 3, 66, 65)
case0051_seg0: features(20, 3, 66, 65)
case0051_seg1: features(20, 3, 66, 65)
case0051_seg2: features(20, 3, 66, 65)
case0051_seg3: features(20, 3, 66, 65)
case0051_seg4: features(20, 3, 66, 65)
case0051_seg5: features(20, 3, 66, 65)
case0052_seg0: features(20, 3, 66, 65)
case0052_seg1: features(20, 3, 66, 65)
case0052_seg2: features(20, 3, 66, 65)
case0052_seg3: features(20, 3, 66, 65)
case0052_seg4: features(20, 3, 66, 65)
case0052_seg5: features(20, 3, 66, 65)
case0052_seg6: features(20, 3, 66, 65)
case0053_seg0: features(20, 3, 66, 65)
case0053_seg1: features(20, 3, 66, 65)
case0053_seg2: features(20, 3, 66, 65)
case0053_seg3: features(20, 3, 66, 65)
case0053_seg4: features(20, 3, 66, 65)
case0053_seg5: features(20, 3, 66, 65)
case0053_seg6: features(20, 3, 66, 65)
case0053_seg7: features(2

# Convert to hdf5

In [11]:
# Convert to hdf5

import torch
import torch.nn.functional as F
import h5py
import numpy as np
from pathlib import Path
from tqdm import tqdm

input_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cylinder_prop/cylinder_prop_segmented')
output = Path('/Volumes/T7/CFDBench/Processed_experiment/cylinder_prop/cylinder_prop_final/cylinder_prop_converted.h5')

t_len = 20
case_folders = sorted([f for f in input_path.iterdir() if f.is_dir()])
num_segments = len(case_folders)

with h5py.File(output, 'w') as f:
    velocity = f.create_dataset('velocity', shape=(num_segments, t_len, 512, 512, 2),
                                dtype='float32', chunks=(1, t_len, 512, 512, 2))
    particles = f.create_dataset('particles', shape=(num_segments, t_len, 512, 512, 1),
                                 dtype='float32', chunks=(1, t_len, 512, 512, 1))

    for i, case_dir in enumerate(tqdm(case_folders)):
        features = np.load(case_dir / 'features.npy')

        u = features[:, 0, :, :]
        v = features[:, 1, :, :]
        mask = features[:, 2, :, :]

        u_t = torch.from_numpy(u).unsqueeze(1)
        v_t = torch.from_numpy(v).unsqueeze(1)
        mask_t = torch.from_numpy(mask).unsqueeze(1)

        # Upsample to 512x512
        u_up = F.interpolate(u_t, size=(512, 512), mode='bilinear', align_corners=True)
        v_up = F.interpolate(v_t, size=(512, 512), mode='bilinear', align_corners=True)
        mask_up = F.interpolate(mask_t, size=(512, 512), mode='nearest')

        # Permute
        velocity_stack = torch.cat([u_up, v_up], dim=1).permute(0, 2, 3, 1).numpy()
        mask_stack = mask_up.permute(0, 2, 3, 1).numpy()

        velocity[i] = velocity_stack
        particles[i] = mask_stack

print('Done')

100%|██████████| 800/800 [01:22<00:00,  9.67it/s]

Done


# Inference results for converted CYLINDER/PROP

In [12]:
# Cylinder/Prop RESULTS
# INFO - 04/06/26 21:58:19 - 0:08:32 - Evaluation Stats (total size = 50)
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# | type         | dim   | size   |   data_loss |   rel l2 |   rel l2 step 1 |   rel l2 step 5 |   rel l2 step 10 |   rel l2 interior |
# +==============+=======+========+=============+==========+=================+=================+==================+===================+
# | incom_ns     | 3     | 50     |    0.298082 |   0.1660 |          0.1506 |          0.1580 |           0.1660 |            0.1612 |
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# | AVE_BY_CLASS | -     | -      |    0.298082 |   0.1660 |          0.1506 |          0.1580 |           0.1660 |            0.1612 |
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# INFO - 04/06/26 21:58:19 - 0:08:32 - Additional Stats for Rel L2 Error:
#     +----------+--------+--------+--------+--------+--------+----------+
#     | type     |   size |   mean |    std |    min |    max |   median |
#     +==========+========+========+========+========+========+==========+
#     | incom_ns |     50 | 0.1660 | 0.0070 | 0.1517 | 0.1769 |   0.1681 |
#     +----------+--------+--------+--------+--------+--------+----------+
# INFO - 04/06/26 21:58:19 - 0:08:32 - Eval | data loss = 0.298082 | rel l2 = 0.166037 | rel l2 step 1 = 0.150598 | rel l2 step 5 = 0.157987 | rel l2 step 10 = 0.166037 | rel l2 interior = 0.161166
# INFO - 04/06/26 21:58:19 - 0:08:32 -  MEM: 0.00 MB


# Slice out a subset first so inference is faster

# import h5py
#
# input_h5 = '/Volumes/T7/CFDBench/Processed_experiment/cylinder_prop/cylinder_prop_final/cylinder_prop_converted.h5'
# output_h5 = '/Volumes/T7/CFDBench/Processed_experiment/cylinder_prop/cylinder_prop_final/cylinder_prop_converted_sliced.h5'
#
# with h5py.File(input_h5, 'r') as f:
#     with h5py.File(output_h5, 'w') as w:
#         for key in f.keys():
#             subset = f[key][:50]
#
#             w.create_dataset(key, data=subset)
#
# print('Done')
#
# with h5py.File(output_h5, 'r') as f:
#     print(f.keys())
#     print(f['velocity'])
#     print(f['particles'])

Done
<KeysViewHDF5 ['particles', 'velocity']>
<HDF5 dataset "velocity": shape (50, 20, 512, 512, 2), type "<f4">
<HDF5 dataset "particles": shape (50, 20, 512, 512, 1), type "<f4">


# Process is repeated for CYLINDER/GEO -> PDEBench INS

In [13]:
# Cylinder/geo
# Let channel 3 be geo mask
# Remove simulations with <20 timesteps
# For those >20 timesteps, split into segments of 20 and concat accordingly

import h5py
import numpy as np
import os

cfd_tube_prop = '/Volumes/T7/CFDBench/cylinder/geo'

for case in sorted(os.listdir(cfd_tube_prop)):
    case_dir = os.path.join(cfd_tube_prop, case)

    if os.path.isdir(case_dir):
        u = np.load(os.path.join(case_dir, 'u.npy'), mmap_mode='r')
        v = np.load(os.path.join(case_dir, 'v.npy'), mmap_mode='r')

        print(f"{case}: u{u.shape}, v{v.shape}")

case0001: u(2000, 64, 64), v(2000, 64, 64)
case0002: u(2000, 64, 64), v(2000, 64, 64)
case0003: u(2000, 64, 64), v(2000, 64, 64)
case0004: u(2000, 64, 64), v(2000, 64, 64)
case0005: u(2000, 64, 64), v(2000, 64, 64)
case0006: u(2000, 64, 64), v(2000, 64, 64)
case0007: u(2000, 64, 64), v(2000, 64, 64)
case0008: u(2000, 64, 64), v(2000, 64, 64)
case0009: u(2000, 64, 64), v(2000, 64, 64)
case0010: u(2000, 64, 64), v(2000, 64, 64)
case0011: u(2000, 64, 64), v(2000, 64, 64)
case0012: u(2000, 64, 64), v(2000, 64, 64)
case0013: u(2000, 64, 64), v(2000, 64, 64)
case0014: u(2000, 64, 64), v(2000, 64, 64)
case0015: u(2000, 64, 64), v(2000, 64, 64)
case0016: u(2000, 64, 64), v(2000, 64, 64)
case0017: u(2000, 64, 64), v(2000, 64, 64)
case0018: u(2000, 64, 64), v(2000, 64, 64)
case0019: u(2000, 64, 64), v(2000, 64, 64)
case0020: u(2000, 64, 64), v(2000, 64, 64)


In [14]:
# Padding
import numpy as np
import json
from pathlib import Path
from tqdm import tqdm
import shutil

def load_json(path):
    with open(path, 'r', encoding='utf8') as f:
        return json.load(f)

input_path = Path('/Volumes/T7/CFDBench/cylinder/geo')
output_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cylinder_geo/cylinder_geo_padded')

for case_dir in tqdm(sorted(input_path.iterdir())):

    u_file = case_dir / "u.npy"
    v_file = case_dir / "v.npy"
    u = np.load(u_file)
    v = np.load(v_file)

    case_params = load_json(case_dir / "case.json")

    mask = np.ones_like(u)

    x_min = case_params["x_min"]
    x_max = case_params["x_max"]
    y_min = case_params["y_min"]
    y_max = case_params["y_max"]
    radius = case_params["radius"]
    case_params["center_x"] = -x_min
    case_params["center_y"] = -y_min
    for key in ["x_min", "x_max", "y_min", "y_max"]:
        del case_params[key]

    height = y_max - y_min
    width = x_max - x_min
    case_params["height"] = height
    case_params["width"] = width

    dx = width / u.shape[2]
    dy = height / u.shape[1]

    for i in range(u.shape[1]):
        for j in range(u.shape[2]):
            x = x_min + j * dx
            y = y_min + i * dy
            if (x - 0.5) ** 2 + (y - 0.5) ** 2 <= radius**2:
                mask[:, i, j] = 0

    # Pad left
    u = np.pad(u, ((0, 0), (0, 0), (1, 0)), mode="constant", constant_values=case_params["vel_in"],)
    v = np.pad(v, ((0, 0), (0, 0), (1, 0)), mode="constant", constant_values=0)
    mask = np.pad(mask, ((0, 0), (0, 0), (1, 0)), mode="constant", constant_values=0)
    # # Pad the top and bottom
    u = np.pad(u, ((0, 0), (1, 1), (0, 0)), mode="constant", constant_values=0)
    v = np.pad(v, ((0, 0), (1, 1), (0, 0)), mode="constant", constant_values=0)
    mask = np.pad(mask, ((0, 0), (1, 1), (0, 0)), mode="constant", constant_values=0)
    features = np.stack([u, v, mask], axis=1)  # (T, 3, h, w)

    case_out_dir = output_path / case_dir.name
    case_out_dir.mkdir(exist_ok=True)

    np.save(case_out_dir / "features.npy", features)
    with open(case_out_dir / "case.json", "w", encoding="utf8") as f:
        json.dump(case_params, f, indent=4)

print('Done')


100%|██████████| 20/20 [00:09<00:00,  2.22it/s]

Done


In [15]:
# Padded check

import h5py
import numpy as np
import os

cfd_tube_prop = '/Volumes/T7/CFDBench/Processed_experiment/cylinder_geo/cylinder_geo_padded'

for case in sorted(os.listdir(cfd_tube_prop)):
    case_dir = os.path.join(cfd_tube_prop, case)

    if os.path.isdir(case_dir):
        features = np.load(os.path.join(case_dir, 'features.npy'), mmap_mode='r')

        print(f"{case}: features{features.shape}")

case0001: features(2000, 3, 66, 65)
case0002: features(2000, 3, 66, 65)
case0003: features(2000, 3, 66, 65)
case0004: features(2000, 3, 66, 65)
case0005: features(2000, 3, 66, 65)
case0006: features(2000, 3, 66, 65)
case0007: features(2000, 3, 66, 65)
case0008: features(2000, 3, 66, 65)
case0009: features(2000, 3, 66, 65)
case0010: features(2000, 3, 66, 65)
case0011: features(2000, 3, 66, 65)
case0012: features(2000, 3, 66, 65)
case0013: features(2000, 3, 66, 65)
case0014: features(2000, 3, 66, 65)
case0015: features(2000, 3, 66, 65)
case0016: features(2000, 3, 66, 65)
case0017: features(2000, 3, 66, 65)
case0018: features(2000, 3, 66, 65)
case0019: features(2000, 3, 66, 65)
case0020: features(2000, 3, 66, 65)


In [16]:
# Convergence
import numpy as np
import shutil
from pathlib import Path
from tqdm import tqdm

input_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cylinder_geo/cylinder_geo_padded')
output_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cylinder_geo/cylinder_geo_convergence')

for case_dir in tqdm(sorted(input_path.iterdir())):

    if not case_dir.is_dir():
        continue

    features_file = case_dir / "features.npy"

    features = np.load(features_file)
    num_timesteps = features.shape[0]

    converged_idx = num_timesteps - 1

    for i in range(converged_idx):
        u_in, v_in = features[i, 0, :, :], features[i, 1, :, :]
        u_out, v_out = features[i+1, 0, :, :], features[i+1, 1, :, :]

        mag_in = np.sqrt(u_in**2 + v_in**2)
        mag_out = np.sqrt(u_out**2 + v_out**2)

        diff = np.abs(mag_in - mag_out).mean()

        if diff < 1e-3:
            converged_idx = i
            break

    case_out_dir = output_path / case_dir.name
    case_out_dir.mkdir(exist_ok=True)

    truncated_features = features[:converged_idx + 1]
    np.save(case_out_dir / "features.npy", truncated_features.astype('float32'))
    shutil.copy(case_dir / "case.json", case_out_dir / "case.json")

print('Done')

100%|██████████| 20/20 [00:05<00:00,  3.73it/s]

Done


In [17]:
# Convergence check

import h5py
import numpy as np
import os

cfd_tube_prop = '/Volumes/T7/CFDBench/Processed_experiment/cylinder_geo/cylinder_geo_convergence'

for case in sorted(os.listdir(cfd_tube_prop)):
    case_dir = os.path.join(cfd_tube_prop, case)

    if os.path.isdir(case_dir):
        features = np.load(os.path.join(case_dir, 'features.npy'), mmap_mode='r')

        print(f"{case}: features{features.shape}")

case0001: features(236, 3, 66, 65)
case0002: features(222, 3, 66, 65)
case0003: features(177, 3, 66, 65)
case0004: features(158, 3, 66, 65)
case0005: features(192, 3, 66, 65)
case0006: features(198, 3, 66, 65)
case0007: features(189, 3, 66, 65)
case0008: features(1, 3, 66, 65)
case0009: features(2000, 3, 66, 65)
case0010: features(2000, 3, 66, 65)
case0011: features(148, 3, 66, 65)
case0012: features(100, 3, 66, 65)
case0013: features(2000, 3, 66, 65)
case0014: features(2000, 3, 66, 65)
case0015: features(155, 3, 66, 65)
case0016: features(100, 3, 66, 65)
case0017: features(81, 3, 66, 65)
case0018: features(286, 3, 66, 65)
case0019: features(321, 3, 66, 65)
case0020: features(404, 3, 66, 65)


In [18]:
# Segmentation
import numpy as np
import shutil
from pathlib import Path
from tqdm import tqdm

input_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cylinder_geo/cylinder_geo_convergence')
output_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cylinder_geo/cylinder_geo_segmented')

t_len = 20

for case_dir in tqdm(sorted(input_path.iterdir())):

    if not case_dir.is_dir():
        continue

    features_file = case_dir / "features.npy"
    json_file = case_dir / "case.json"

    features = np.load(features_file)
    num_timesteps = features.shape[0]

    num_segments = num_timesteps // t_len

    if num_segments == 0:
        continue

    for i in range(num_segments):
        start = i * t_len
        end = start + t_len

        segment = features[start:end]

        segment_name = f"{case_dir.name}_seg{i}"
        segment_out_dir = output_path / segment_name
        segment_out_dir.mkdir(exist_ok=True)

        np.save(segment_out_dir / "features.npy", segment.astype('float32'))
        shutil.copy(json_file, segment_out_dir / "case.json")

print('Done')

100%|██████████| 20/20 [00:01<00:00, 15.85it/s]

Done


In [19]:
# Segmentation check

import h5py
import numpy as np
import os

cfd_tube_prop = '/Volumes/T7/CFDBench/Processed_experiment/cylinder_geo/cylinder_geo_segmented'

for case in sorted(os.listdir(cfd_tube_prop)):
    case_dir = os.path.join(cfd_tube_prop, case)

    if os.path.isdir(case_dir):
        features = np.load(os.path.join(case_dir, 'features.npy'), mmap_mode='r')

        print(f"{case}: features{features.shape}")

case0001_seg0: features(20, 3, 66, 65)
case0001_seg1: features(20, 3, 66, 65)
case0001_seg10: features(20, 3, 66, 65)
case0001_seg2: features(20, 3, 66, 65)
case0001_seg3: features(20, 3, 66, 65)
case0001_seg4: features(20, 3, 66, 65)
case0001_seg5: features(20, 3, 66, 65)
case0001_seg6: features(20, 3, 66, 65)
case0001_seg7: features(20, 3, 66, 65)
case0001_seg8: features(20, 3, 66, 65)
case0001_seg9: features(20, 3, 66, 65)
case0002_seg0: features(20, 3, 66, 65)
case0002_seg1: features(20, 3, 66, 65)
case0002_seg10: features(20, 3, 66, 65)
case0002_seg2: features(20, 3, 66, 65)
case0002_seg3: features(20, 3, 66, 65)
case0002_seg4: features(20, 3, 66, 65)
case0002_seg5: features(20, 3, 66, 65)
case0002_seg6: features(20, 3, 66, 65)
case0002_seg7: features(20, 3, 66, 65)
case0002_seg8: features(20, 3, 66, 65)
case0002_seg9: features(20, 3, 66, 65)
case0003_seg0: features(20, 3, 66, 65)
case0003_seg1: features(20, 3, 66, 65)
case0003_seg2: features(20, 3, 66, 65)
case0003_seg3: features

In [20]:
# Convert to hdf5

import torch
import torch.nn.functional as F
import h5py
import numpy as np
from pathlib import Path
from tqdm import tqdm

input_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cylinder_geo/cylinder_geo_segmented')
output = Path('/Volumes/T7/CFDBench/Processed_experiment/cylinder_geo/cylinder_geo_final/cylinder_geo_converted.h5')

t_len = 20
case_folders = sorted([f for f in input_path.iterdir() if f.is_dir()])
num_segments = len(case_folders)

with h5py.File(output, 'w') as f:
    velocity = f.create_dataset('velocity', shape=(num_segments, t_len, 512, 512, 2),
                                dtype='float32', chunks=(1, t_len, 512, 512, 2))
    particles = f.create_dataset('particles', shape=(num_segments, t_len, 512, 512, 1),
                                 dtype='float32', chunks=(1, t_len, 512, 512, 1))

    for i, case_dir in enumerate(tqdm(case_folders)):
        features = np.load(case_dir / 'features.npy')

        u = features[:, 0, :, :]
        v = features[:, 1, :, :]
        mask = features[:, 2, :, :]

        u_t = torch.from_numpy(u).unsqueeze(1)
        v_t = torch.from_numpy(v).unsqueeze(1)
        mask_t = torch.from_numpy(mask).unsqueeze(1)

        # Upsample to 512x512
        u_up = F.interpolate(u_t, size=(512, 512), mode='bilinear', align_corners=True)
        v_up = F.interpolate(v_t, size=(512, 512), mode='bilinear', align_corners=True)
        mask_up = F.interpolate(mask_t, size=(512, 512), mode='nearest')

        # Permute
        velocity_stack = torch.cat([u_up, v_up], dim=1).permute(0, 2, 3, 1).numpy()
        mask_stack = mask_up.permute(0, 2, 3, 1).numpy()

        velocity[i] = velocity_stack
        particles[i] = mask_stack

print('Done')

100%|██████████| 542/542 [00:56<00:00,  9.63it/s]

Done


# Inference results for converted CYLINDER/GEO

In [21]:
# Cylinder/Geo RESULTS

# INFO - 04/06/26 22:08:31 - 0:06:31 - Evaluation Stats (total size = 50)
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# | type         | dim   | size   |   data_loss |   rel l2 |   rel l2 step 1 |   rel l2 step 5 |   rel l2 step 10 |   rel l2 interior |
# +==============+=======+========+=============+==========+=================+=================+==================+===================+
# | incom_ns     | 3     | 50     |    0.333881 |   0.1669 |          0.1516 |          0.1590 |           0.1669 |            0.1592 |
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# | AVE_BY_CLASS | -     | -      |    0.333881 |   0.1669 |          0.1516 |          0.1590 |           0.1669 |            0.1592 |
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# INFO - 04/06/26 22:08:31 - 0:06:31 - Additional Stats for Rel L2 Error:
#     +----------+--------+--------+--------+--------+--------+----------+
#     | type     |   size |   mean |    std |    min |    max |   median |
#     +==========+========+========+========+========+========+==========+
#     | incom_ns |     50 | 0.1669 | 0.0097 | 0.1489 | 0.1854 |   0.1665 |
#     +----------+--------+--------+--------+--------+--------+----------+
# INFO - 04/06/26 22:08:31 - 0:06:31 - Eval | data loss = 0.333881 | rel l2 = 0.166919 | rel l2 step 1 = 0.151572 | rel l2 step 5 = 0.158988 | rel l2 step 10 = 0.166919 | rel l2 interior = 0.159155
# INFO - 04/06/26 22:08:31 - 0:06:31 -  MEM: 0.00 MB

# Slice out a subset first so inference is faster

# import h5py
#
# input_h5 = '/Volumes/T7/CFDBench/Processed_experiment/cylinder_geo/cylinder_geo_final/cylinder_geo_converted.h5'
# output_h5 = '/Volumes/T7/CFDBench/Processed_experiment/cylinder_geo/cylinder_geo_final/cylinder_geo_converted_sliced.h5'
#
# with h5py.File(input_h5, 'r') as f:
#     with h5py.File(output_h5, 'w') as w:
#         for key in f.keys():
#             subset = f[key][:50]
#
#             w.create_dataset(key, data=subset)
#
# print('Done')
#
# with h5py.File(output_h5, 'r') as f:
#     print(f.keys())
#     print(f['velocity'])
#     print(f['particles'])

Done
<KeysViewHDF5 ['particles', 'velocity']>
<HDF5 dataset "velocity": shape (50, 20, 512, 512, 2), type "<f4">
<HDF5 dataset "particles": shape (50, 20, 512, 512, 1), type "<f4">


# Process is repeated for CYLINDER/BC -> PDEBench INS

In [22]:
# Cylinder/bc
# Let channel 3 be geo mask
# Remove simulations with <20 timesteps
# For those >20 timesteps, split into segments of 20 and concat accordingly

import h5py
import numpy as np
import os

cfd_tube_prop = '/Volumes/T7/CFDBench/cylinder/bc'

for case in sorted(os.listdir(cfd_tube_prop)):
    case_dir = os.path.join(cfd_tube_prop, case)

    if os.path.isdir(case_dir):
        u = np.load(os.path.join(case_dir, 'u.npy'), mmap_mode='r')
        v = np.load(os.path.join(case_dir, 'v.npy'), mmap_mode='r')

        print(f"{case}: u{u.shape}, v{v.shape}")

case0000: u(620, 64, 64), v(620, 64, 64)
case0001: u(1000, 64, 64), v(1000, 64, 64)
case0002: u(1000, 64, 64), v(1000, 64, 64)
case0003: u(1000, 64, 64), v(1000, 64, 64)
case0004: u(1000, 64, 64), v(1000, 64, 64)
case0005: u(1000, 64, 64), v(1000, 64, 64)
case0006: u(1000, 64, 64), v(1000, 64, 64)
case0007: u(1000, 64, 64), v(1000, 64, 64)
case0008: u(1000, 64, 64), v(1000, 64, 64)
case0009: u(1000, 64, 64), v(1000, 64, 64)
case0010: u(1000, 64, 64), v(1000, 64, 64)
case0011: u(1000, 64, 64), v(1000, 64, 64)
case0012: u(1000, 64, 64), v(1000, 64, 64)
case0013: u(1000, 64, 64), v(1000, 64, 64)
case0014: u(1000, 64, 64), v(1000, 64, 64)
case0015: u(1000, 64, 64), v(1000, 64, 64)
case0016: u(1000, 64, 64), v(1000, 64, 64)
case0017: u(1000, 64, 64), v(1000, 64, 64)
case0018: u(1000, 64, 64), v(1000, 64, 64)
case0019: u(1000, 64, 64), v(1000, 64, 64)
case0020: u(1000, 64, 64), v(1000, 64, 64)
case0021: u(1000, 64, 64), v(1000, 64, 64)
case0022: u(1000, 64, 64), v(1000, 64, 64)
case0023: u(1

In [24]:
# Padding
import numpy as np
import json
from pathlib import Path
from tqdm import tqdm
import shutil

def load_json(path):
    with open(path, 'r', encoding='utf8') as f:
        return json.load(f)

input_path = Path('/Volumes/T7/CFDBench/cylinder/bc')
output_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cylinder_bc/cylinder_bc_padded')

for case_dir in tqdm(sorted(input_path.iterdir())):

    if not case_dir.is_dir():
        continue

    u_file = case_dir / "u.npy"
    v_file = case_dir / "v.npy"
    u = np.load(u_file)
    v = np.load(v_file)

    case_params = load_json(case_dir / "case.json")

    mask = np.ones_like(u)

    x_min = case_params["x_min"]
    x_max = case_params["x_max"]
    y_min = case_params["y_min"]
    y_max = case_params["y_max"]
    radius = case_params["radius"]
    case_params["center_x"] = -x_min
    case_params["center_y"] = -y_min
    for key in ["x_min", "x_max", "y_min", "y_max"]:
        del case_params[key]

    height = y_max - y_min
    width = x_max - x_min
    case_params["height"] = height
    case_params["width"] = width

    dx = width / u.shape[2]
    dy = height / u.shape[1]

    for i in range(u.shape[1]):
        for j in range(u.shape[2]):
            x = x_min + j * dx
            y = y_min + i * dy
            if (x - 0.5) ** 2 + (y - 0.5) ** 2 <= radius**2:
                mask[:, i, j] = 0

    # Pad left
    u = np.pad(u, ((0, 0), (0, 0), (1, 0)), mode="constant", constant_values=case_params["vel_in"],)
    v = np.pad(v, ((0, 0), (0, 0), (1, 0)), mode="constant", constant_values=0)
    mask = np.pad(mask, ((0, 0), (0, 0), (1, 0)), mode="constant", constant_values=0)
    # # Pad the top and bottom
    u = np.pad(u, ((0, 0), (1, 1), (0, 0)), mode="constant", constant_values=0)
    v = np.pad(v, ((0, 0), (1, 1), (0, 0)), mode="constant", constant_values=0)
    mask = np.pad(mask, ((0, 0), (1, 1), (0, 0)), mode="constant", constant_values=0)
    features = np.stack([u, v, mask], axis=1)  # (T, 3, h, w)

    case_out_dir = output_path / case_dir.name
    case_out_dir.mkdir(exist_ok=True)

    np.save(case_out_dir / "features.npy", features)
    with open(case_out_dir / "case.json", "w", encoding="utf8") as f:
        json.dump(case_params, f, indent=4)

print('Done')


100%|██████████| 51/51 [00:11<00:00,  4.51it/s]

Done


In [25]:
# Padded check

import h5py
import numpy as np
import os

cfd_tube_prop = '/Volumes/T7/CFDBench/Processed_experiment/cylinder_bc/cylinder_bc_padded'

for case in sorted(os.listdir(cfd_tube_prop)):
    case_dir = os.path.join(cfd_tube_prop, case)

    if os.path.isdir(case_dir):
        features = np.load(os.path.join(case_dir, 'features.npy'), mmap_mode='r')

        print(f"{case}: features{features.shape}")

case0000: features(620, 3, 66, 65)
case0001: features(1000, 3, 66, 65)
case0002: features(1000, 3, 66, 65)
case0003: features(1000, 3, 66, 65)
case0004: features(1000, 3, 66, 65)
case0005: features(1000, 3, 66, 65)
case0006: features(1000, 3, 66, 65)
case0007: features(1000, 3, 66, 65)
case0008: features(1000, 3, 66, 65)
case0009: features(1000, 3, 66, 65)
case0010: features(1000, 3, 66, 65)
case0011: features(1000, 3, 66, 65)
case0012: features(1000, 3, 66, 65)
case0013: features(1000, 3, 66, 65)
case0014: features(1000, 3, 66, 65)
case0015: features(1000, 3, 66, 65)
case0016: features(1000, 3, 66, 65)
case0017: features(1000, 3, 66, 65)
case0018: features(1000, 3, 66, 65)
case0019: features(1000, 3, 66, 65)
case0020: features(1000, 3, 66, 65)
case0021: features(1000, 3, 66, 65)
case0022: features(1000, 3, 66, 65)
case0023: features(1000, 3, 66, 65)
case0024: features(1000, 3, 66, 65)
case0025: features(1000, 3, 66, 65)
case0026: features(1000, 3, 66, 65)
case0027: features(1000, 3, 6

In [26]:
# Convergence
import numpy as np
import shutil
from pathlib import Path
from tqdm import tqdm

input_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cylinder_bc/cylinder_bc_padded')
output_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cylinder_bc/cylinder_bc_convergence')

for case_dir in tqdm(sorted(input_path.iterdir())):

    if not case_dir.is_dir():
        continue

    features_file = case_dir / "features.npy"

    features = np.load(features_file)
    num_timesteps = features.shape[0]

    converged_idx = num_timesteps - 1

    for i in range(converged_idx):
        u_in, v_in = features[i, 0, :, :], features[i, 1, :, :]
        u_out, v_out = features[i+1, 0, :, :], features[i+1, 1, :, :]

        mag_in = np.sqrt(u_in**2 + v_in**2)
        mag_out = np.sqrt(u_out**2 + v_out**2)

        diff = np.abs(mag_in - mag_out).mean()

        if diff < 1e-3:
            converged_idx = i
            break

    case_out_dir = output_path / case_dir.name
    case_out_dir.mkdir(exist_ok=True)

    truncated_features = features[:converged_idx + 1]
    np.save(case_out_dir / "features.npy", truncated_features.astype('float32'))
    shutil.copy(case_dir / "case.json", case_out_dir / "case.json")

print('Done')

100%|██████████| 50/50 [00:08<00:00,  5.72it/s]

Done


In [27]:
# Convergence check

import h5py
import numpy as np
import os

cfd_tube_prop = '/Volumes/T7/CFDBench/Processed_experiment/cylinder_bc/cylinder_bc_convergence'

for case in sorted(os.listdir(cfd_tube_prop)):
    case_dir = os.path.join(cfd_tube_prop, case)

    if os.path.isdir(case_dir):
        features = np.load(os.path.join(case_dir, 'features.npy'), mmap_mode='r')

        print(f"{case}: features{features.shape}")

case0000: features(1, 3, 66, 65)
case0001: features(4, 3, 66, 65)
case0002: features(8, 3, 66, 65)
case0003: features(12, 3, 66, 65)
case0004: features(18, 3, 66, 65)
case0005: features(33, 3, 66, 65)
case0006: features(61, 3, 66, 65)
case0007: features(92, 3, 66, 65)
case0008: features(137, 3, 66, 65)
case0009: features(184, 3, 66, 65)
case0010: features(206, 3, 66, 65)
case0011: features(220, 3, 66, 65)
case0012: features(229, 3, 66, 65)
case0013: features(236, 3, 66, 65)
case0014: features(240, 3, 66, 65)
case0015: features(244, 3, 66, 65)
case0016: features(248, 3, 66, 65)
case0017: features(255, 3, 66, 65)
case0018: features(1000, 3, 66, 65)
case0019: features(1000, 3, 66, 65)
case0020: features(1000, 3, 66, 65)
case0021: features(1000, 3, 66, 65)
case0022: features(1000, 3, 66, 65)
case0023: features(1000, 3, 66, 65)
case0024: features(1000, 3, 66, 65)
case0025: features(1000, 3, 66, 65)
case0026: features(1000, 3, 66, 65)
case0027: features(1000, 3, 66, 65)
case0028: features(10

In [28]:
# Segmentation
import numpy as np
import shutil
from pathlib import Path
from tqdm import tqdm

input_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cylinder_bc/cylinder_bc_convergence')
output_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cylinder_bc/cylinder_bc_segmented')

t_len = 20

for case_dir in tqdm(sorted(input_path.iterdir())):

    if not case_dir.is_dir():
        continue

    features_file = case_dir / "features.npy"
    json_file = case_dir / "case.json"

    features = np.load(features_file)
    num_timesteps = features.shape[0]

    num_segments = num_timesteps // t_len

    if num_segments == 0:
        continue

    for i in range(num_segments):
        start = i * t_len
        end = start + t_len

        segment = features[start:end]

        segment_name = f"{case_dir.name}_seg{i}"
        segment_out_dir = output_path / segment_name
        segment_out_dir.mkdir(exist_ok=True)

        np.save(segment_out_dir / "features.npy", segment.astype('float32'))
        shutil.copy(json_file, segment_out_dir / "case.json")

print('Done')

100%|██████████| 50/50 [00:03<00:00, 12.59it/s]

Done


In [29]:
# Convert to hdf5

import torch
import torch.nn.functional as F
import h5py
import numpy as np
from pathlib import Path
from tqdm import tqdm

input_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cylinder_bc/cylinder_bc_segmented')
output = Path('/Volumes/T7/CFDBench/Processed_experiment/cylinder_bc/cylinder_bc_final/cylinder_bc_converted.h5')

t_len = 20
case_folders = sorted([f for f in input_path.iterdir() if f.is_dir()])
num_segments = len(case_folders)

with h5py.File(output, 'w') as f:
    velocity = f.create_dataset('velocity', shape=(num_segments, t_len, 512, 512, 2),
                                dtype='float32', chunks=(1, t_len, 512, 512, 2))
    particles = f.create_dataset('particles', shape=(num_segments, t_len, 512, 512, 1),
                                 dtype='float32', chunks=(1, t_len, 512, 512, 1))

    for i, case_dir in enumerate(tqdm(case_folders)):
        features = np.load(case_dir / 'features.npy')

        u = features[:, 0, :, :]
        v = features[:, 1, :, :]
        mask = features[:, 2, :, :]

        u_t = torch.from_numpy(u).unsqueeze(1)
        v_t = torch.from_numpy(v).unsqueeze(1)
        mask_t = torch.from_numpy(mask).unsqueeze(1)

        # Upsample to 512x512
        u_up = F.interpolate(u_t, size=(512, 512), mode='bilinear', align_corners=True)
        v_up = F.interpolate(v_t, size=(512, 512), mode='bilinear', align_corners=True)
        mask_up = F.interpolate(mask_t, size=(512, 512), mode='nearest')

        # Permute
        velocity_stack = torch.cat([u_up, v_up], dim=1).permute(0, 2, 3, 1).numpy()
        mask_stack = mask_up.permute(0, 2, 3, 1).numpy()

        velocity[i] = velocity_stack
        particles[i] = mask_stack

print('Done')

100%|██████████| 1714/1714 [03:51<00:00,  7.40it/s]


Done


# Inference results for converted CYLINDER/BC

In [30]:
# Cylinder/bc RESULTS

# INFO - 04/06/26 22:23:40 - 0:06:29 - Evaluation Stats (total size = 50)
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# | type         | dim   | size   |   data_loss |   rel l2 |   rel l2 step 1 |   rel l2 step 5 |   rel l2 step 10 |   rel l2 interior |
# +==============+=======+========+=============+==========+=================+=================+==================+===================+
# | incom_ns     | 3     | 50     |    0.319173 |   0.1632 |          0.1484 |          0.1555 |           0.1632 |            0.1589 |
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# | AVE_BY_CLASS | -     | -      |    0.319173 |   0.1632 |          0.1484 |          0.1555 |           0.1632 |            0.1589 |
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# INFO - 04/06/26 22:23:40 - 0:06:29 - Additional Stats for Rel L2 Error:
#     +----------+--------+--------+--------+--------+--------+----------+
#     | type     |   size |   mean |    std |    min |    max |   median |
#     +==========+========+========+========+========+========+==========+
#     | incom_ns |     50 | 0.1632 | 0.0117 | 0.1357 | 0.1858 |   0.1638 |
#     +----------+--------+--------+--------+--------+--------+----------+
# INFO - 04/06/26 22:23:40 - 0:06:29 - Eval | data loss = 0.319173 | rel l2 = 0.163191 | rel l2 step 1 = 0.148419 | rel l2 step 5 = 0.155528 | rel l2 step 10 = 0.163191 | rel l2 interior = 0.158949
# INFO - 04/06/26 22:23:40 - 0:06:29 -  MEM: 0.00 MB

# Slice out a subset first so inference is faster

# import h5py
#
# input_h5 = '/Volumes/T7/CFDBench/Processed_experiment/cylinder_bc/cylinder_bc_final/cylinder_bc_converted.h5'
# output_h5 = '/Volumes/T7/CFDBench/Processed_experiment/cylinder_bc/cylinder_bc_final/cylinder_bc_converted_sliced.h5'
#
# with h5py.File(input_h5, 'r') as f:
#     with h5py.File(output_h5, 'w') as w:
#         for key in f.keys():
#             subset = f[key][:50]
#
#             w.create_dataset(key, data=subset)
#
# print('Done')
#
# with h5py.File(output_h5, 'r') as f:
#     print(f.keys())
#     print(f['velocity'])
#     print(f['particles'])

Done
<KeysViewHDF5 ['particles', 'velocity']>
<HDF5 dataset "velocity": shape (50, 20, 512, 512, 2), type "<f4">
<HDF5 dataset "particles": shape (50, 20, 512, 512, 1), type "<f4">
